In [3]:
# Search_Names_Probability.py
# 5/28/2025

from bs4 import BeautifulSoup 
import datetime
from datetime import datetime as dt
from datetime import date 
from datetime import timedelta
import time
import re
from pandas import read_csv
import pandas as pd
# from selenium import webdriver
from seleniumwire import webdriver
from selenium.webdriver.common.by import By
import json
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
import sys
import random
from zenrows import ZenRowsClient
import requests
import math
import numpy as np
import urllib3
import inspect
from geopy.distance import distance
# from IPython.display import display


# def maximize_screen():
#     from IPython.display import display, HTML
#     display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

def left(s, amount):
    return s[:amount]
def right(s, amount):
    return s[-amount:]
def mid(s, offset, amount):
    return s[offset:offset+amount]

def maximize_screen():
    from IPython.display import display, HTML
    display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

def timestamp():
    today=str(datetime.date.today()).split('-')
    year=today[0]
    month=today[1]
    day=today[2]
    a=str(datetime.datetime.now())
    time_split=re.split(':| ',a)
    hour=time_split[1]
    timestamp=year+month+day+hour
    return timestamp

def convert_date_z(date): 
    date_t= pd.to_datetime(date, utc=True)
    date_out=date_t.dt.strftime("%m/%d/%Y")
    return date_out

def convert_datetime_t(event_date_t):
    no_time='2000-01-01T00:00:00'
    no_time_dt=dt.strptime(no_time, "%Y-%m-%dT%H:%M:%S") 
    event_date_t = event_date_t.fillna(no_time)
    event_date_dt=pd.to_datetime(event_date_t, format="%Y-%m-%dT%H:%M:%S")
    event_date_index=event_date_dt.dt.strftime("%Y%m%d")
    event_date =event_date_dt.dt.strftime("%m/%d/%Y")
    event_date_rnd_dt = event_date_dt + timedelta(minutes=30)
    event_time_index=event_date_rnd_dt.dt.strftime('%H')  
    event_time = event_date_dt.dt.strftime('%H:%M')
    return     event_date,event_date_index,event_time,event_time_index 



def random_sleep(call_count): 
    global sleep_time_pause,sleep_range
    if call_count%100==0:
        sleep_time =sleep_time_pause
        now_date = datetime.datetime.now() + timedelta(seconds=sleep_time)
        sleep_end_date=now_date.strftime("%m/%d/%Y %H:%M:%S")    
        print(f"\t\t\tSleeping for {sleep_time:.0f} seconds • Sleeping till: {sleep_end_date}")
    else:
        sleep_time = random.uniform(sleep_range[0],sleep_range[1])
    return sleep_time 

def get_total_cemeteries(url):
    global my_header
    response = requests.request("GET", url, headers=my_header) 
    status_code=response.status_code
    url_source = response.text 
    
    beautifulSoupText = BeautifulSoup(url_source, 'html.parser') 
    specific_tag_text = beautifulSoupText.get_text()
    capture_json=json.loads(specific_tag_text) 
    total_cemeteries=capture_json['data']['cemeteries']['total']
    return total_cemeteries

def get_grave_count(url):
    response = requests.request("GET", url, headers=my_header) 
    status_code=response.status_code
    url_source = response.text 
    beautifulSoupText = BeautifulSoup(url_source, 'html.parser') 
    specific_tag_text = beautifulSoupText.get_text()
    capture_json=json.loads(specific_tag_text) 
    grave_count=capture_json['total']
    return capture_json,grave_count
    
def extract_grave_data(capture_json):
    global field_list_set,field_list
    collection_df=pd.json_normalize(capture_json['collection'])
    collection_columns_set=set(list(collection_df))
    missing_columns=field_list_set-collection_columns_set
    collection_df[list(missing_columns)]=np.nan
    grave_frame_df=collection_df[field_list]
    return grave_frame_df

def clean_dataframe(graves_df):
    global field_list
    for column in field_list:
        graves_df[column]= graves_df[column].astype(str).replace(r"\,", ' * ', regex=True).str.strip()
        graves_df[column]= graves_df[column].astype(str).replace(r"\'", '', regex=True).str.strip()
    delete_list=[r'\"',r"\'",r'\?',r'\[',r'\]']
    for item in delete_list:
        for column in field_list:
            graves_df[column]= graves_df[column].astype(str).replace(item, ' ', regex=True).str.strip()
    return graves_df


def get_cemetery_cluster(name_location):
    global jewish_cemetery_df,non_jewish_cemetery_df,county_coordinates_df
    county_distance_df=pd.DataFrame([])
    # name_location=[name_lat,name_lon]
    for _, row in county_coordinates_df.iterrows():
        county_distance=distance(name_location, (row['county_lat'], row['county_lon'])).miles
        county_distance_line_df=pd.DataFrame([row['county_id'],row['county_name'],row['state_name'],county_distance]).T
        county_distance_line_df.columns=['county_id','county_name','state_name','distance']
        county_distance_df=pd.concat([county_distance_df,county_distance_line_df])
    cluster_county_id_df=county_distance_df[['county_id','county_name','state_name']][county_distance_df['distance']<=jewish_radius]
    matched_cemeteries_df_header=['type','cemetery_id','cemetery_name','grave_count','city','state','stateAbbreviation',\
                              'county','state_id','county_id','cemetery_lat','cemetery_lon','County_Cemetery_Counts']
    jewish_cemetery_df=pd.merge(
        jewish_cemetery_df,    
        cluster_county_id_df,
        right_on='county_id',             
        left_on='county_id',
        how='inner'  # or 'left', 'right', 'outer' depending on what you want
    )
    jewish_cemetery_df['type']='Jewish'
    
    non_jewish_cemetery_df=pd.merge(
        non_jewish_cemetery_df,    
        cluster_county_id_df,
        right_on='county_id',             
        left_on='county_id',
        how='inner'  # or 'left', 'right', 'outer' depending on what you want
    )
    non_jewish_cemetery_df['type']='Non_Jewish'
    matched_cemeteries_df=pd.DataFrame([])
    matched_cemeteries_df = non_jewish_cemetery_df._append(jewish_cemetery_df, ignore_index = True) 
    matched_cemeteries_df=matched_cemeteries_df[matched_cemeteries_df_header]
    return matched_cemeteries_df,cluster_county_id_df

my_header = {
  'priority': 'u=1, i',
  'sec-ch-ua': '"Google Chrome";v="135", "Not-A.Brand";v="8", "Chromium";v="135"',
  'sec-ch-ua-platform': '"Windows"',
  'sec-fetch-mode': 'cors',
  'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/135.0.0.0 Safari/537.36',
  'Cookie': '__cf_bm=O55WB_mmMt72i8KoQuzWliizNTb6J7zg2RVHXqbn.vc-1745620049-1.0.1.1-IDSlPNW4kRjBMAr5n8M2obOoGp9GAod_BHbB997Ze2F0YUm2DvAFh5TaijsQhPWR0iyIyFoaSy.p1YDke07ZttHHC83SJpQCEBFmaeImKIc; _cfuvid=zhvP.IczaAr9sd4YVjWjTqBayGxsei_Ntl0ANgSCCBg-1745620049033-0.0.1.1-604800000; ipLoc=us; preferredLanguage=en'
}

#*************** Append  *******************************
append=0
# append_flag=input('To Append Press 1')
# if append_flag=='1':append=1
#*************** Append  *******************************

maximize_screen()
timestamp=timestamp()
collection_date0=datetime.datetime.now().strftime("%m/%d/%Y")
collection_date_dt=pd.to_datetime(collection_date0)
collection_date=collection_date0

start_time=datetime.datetime.now()
event_time=start_time
blank_line=['                                                                                                                                                                               ']

process_name='Extract Jewish ABMC Graves '
print('**Starting *** ',process_name,' *** , Time:',start_time,' **\n')
DirMain="C:\\Users\\jrozm\\OneDrive\\Benjamin\\"



fileLastIn="Last_Names_List"
fileFirstIn="First_names_List"
fileJewishIn="Jewish_Cluster_Cemetery_List"
fileNonJewishIn="NonJewish_Cluster_Cemetery_List"
fileCountyIn="Jewish_Counties_Coordinates"


fileLastOut="Last_Name_Counts_Summary"
fileFirstOut="First_Name_Counts_Summary"


fileInJ=DirMain+"Data\\"+fileJewishIn+".csv"
fileInN=DirMain+"Data\\"+fileNonJewishIn+".csv"
fileInCounty=DirMain+"Data\\"+fileCountyIn+".csv"

fileLastNamesIn=DirMain+"Analysis\\"+fileLastIn+".csv"
fileFirstNamesIn=DirMain+"Analysis\\"+fileFirstIn+".csv"

fileLastNamesOut=DirMain+"Analysis\\SearchResults\\"+fileLastOut+"_"+timestamp+".csv"
fileLastNamesOut0=DirMain+"Analysis\\SearchResults\\"+fileLastOut+".csv"
fileLastNamesOutInt=DirMain+"Analysis\\SearchResults\\"+fileLastOut+"_Interim.csv"

fileFirstNamesOut=DirMain+"Analysis\\SearchResults\\"+fileFirstOut+"_"+timestamp+".csv"
fileFirstNamesOut0=DirMain+"Analysis\\SearchResults\\"+fileFirstOut+".csv"
fileFirstNamesOutInt=DirMain+"Analysis\\SearchResults\\"+fileFirstOut+"_Interim.csv"


jewish_cemetery_df=pd.read_csv(fileInJ)
non_jewish_cemetery_df=pd.read_csv(fileInN)
county_coordinates_df=pd.read_csv(fileInCounty)

cemetery_field_list=['cemetery_id','grave_count','cemetery_name','city','county	state','stateAbbreviation','county_id','state_id','cemetery_lat','cemetery_lon']

grave_df_header_final=['type','lastName','firstName','fullName','maidenName','birthYear','deathYear','birthDate','deathDate','cemeteryName',\
                       'cemeteryId','cityName','stateName','distance','Parents','Children','Siblings','Spouses']
summary_df_header=['Name','City','State','Total_Matched_Graves','Jewish_Graves','Non_Jewish_Graves','Jewish_Graves_Percent','Jewish_Radius','Non_Jewish_Radius']
matched_cemeteries_df_header=['type','cemetery_id','cemetery_name','distance','city','state','state_id','county_id','lat','lon','grave_count']

last_names_df=pd.read_csv(fileLastNamesIn)

last_names_df["last_name"]=last_names_df["last_name"].str.lower()
last_name_list=last_names_df["last_name"]
last_name_lat_list=last_names_df["lat"]
last_name_lon_list=last_names_df["lon"]
last_name_city_list=last_names_df["City"]
last_name_state_list=last_names_df["State"]

first_names_df=pd.read_csv(fileFirstNamesIn)

first_name_list=first_names_df['first_name'].str.lower()
first_name_lat_list=first_names_df["lat"]
first_name_lon_list=first_names_df["lon"]
first_name_city_list=first_names_df["City"]
first_name_state_list=first_names_df["State"]


field_list=['lastName','firstName','fullName','birthYear','deathYear','birthMonth','deathMonth','birthDate','deathDate',\
    'cemeteryName','cemeteryId','cityName','stateName','maidenName','Parents','Children', 'Siblings','Spouses']
field_list_set=set(field_list)
if append==1:print('**** Appending ****')    
if append!=1:graves_df=pd.DataFrame([])  

death_year=1940
birth_year=1920
search_type='death'
search_type='birth'


if search_type=='death':
    url0 = "https://www.findagrave.com/memorial/search?ajax=true&skip="
    url1 = "&limit=100&firstname=&middlename=&lastname="
    url2 ="&birthyear=&birthyearfilter=&deathyear="+str(death_year)+"&deathyearfilter=before&locationId="
    url3="&includeMaidenName=false"
    
    url0_f = "https://www.findagrave.com/memorial/search?ajax=true&skip="
    url1_f = "&limit=100&firstname="
    url2_f ="&middlename=&lastname=&exactName=true&birthyear=&birthyearfilter=&deathyear="+str(death_year)+"&deathyearfilter=before&locationId="
    url3_f="&includeMaidenName=false"
    search_year=death_year
else:
    url0 = "https://www.findagrave.com/memorial/search?ajax=true&skip="
    url1 = "&limit=100&firstname=&middlename=&lastname="
    url2 ="&birthyear="+str(birth_year)+"&birthyearfilter=before&deathyear=&deathyearfilter=&locationId="
    url3="&includeMaidenName=false"
    
    url0_f = "https://www.findagrave.com/memorial/search?ajax=true&skip="
    url1_f = "&limit=100&firstname="
    url2_f ="&middlename=&lastname=&exactName=true&birthyear="+str(birth_year)+"&birthyearfilter=before&deathyear=&deathyearfilter=&locationId="
    url3_f="&includeMaidenName=false"
    search_year=birth_year
    

summary_df=pd.DataFrame([])
sleep_range=[3,8]
sleep_time_pause=1*60

jewish_radius=25
non_jewish_radius=25
get_counter=0
last_name_count=0    


last_name_counts_df=pd.DataFrame([])
for last_name in last_name_list:
    last_name_lat=last_name_lat_list.iloc[last_name_count]
    last_name_lon=last_name_lon_list.iloc[last_name_count]
    last_name_state=last_name_state_list.iloc[last_name_count]
    last_name_city=last_name_city_list.iloc[last_name_count]
    last_name_count+=1
    last_name_location=[last_name_lat,last_name_lon]
    matched_cemeteries_df,county_list=get_cemetery_cluster(last_name_location)   
    county_id_list=county_list['county_id']

    total_county_count=len(county_id_list)
    county_search_count=0
    last_name_df=pd.DataFrame([])
    for last_name in last_name_list:
        print(f"•••• Searching: {last_name} • Search By: {search_type} • Search Before: {search_year}")
        last_jewish_graves_last_name=0
        last_non_jewish_graves_last_name =0
        county_count=0
        for county_id in county_id_list:
            county_name=county_list['county_name'].iloc[county_count]
            state_name=county_list['state_name'].iloc[county_count]
            county_count+=1
            frame_count=0
            grave_count=100000
            grave_frame_count=0
            frame_count=0
            while grave_frame_count<grave_count:
                frame_count+=1
                url=url0+str(grave_frame_count)+url1+last_name+url2+county_id+url3
                capture_json,grave_count=get_grave_count(url)
                last_name_frame_df_raw=extract_grave_data(capture_json)
                grave_frame_count+=len(last_name_frame_df_raw) 
                last_name_frame_df_raw['cemeteryId']=last_name_frame_df_raw['cemeteryId'].fillna(0).astype(int)
                last_name_frame_df_raw['lastName']=last_name_frame_df_raw['lastName'].astype(str).str.lower()
                last_name_frame_df_raw['firstName']=last_name_frame_df_raw['firstName'].astype(str).str.lower()
                last_name_frame_df_raw = pd.merge(
                        last_name_frame_df_raw,    
                        matched_cemeteries_df,
                        right_on='cemetery_id',             
                        left_on='cemeteryId',
                        how='inner'  # or 'left', 'right', 'outer' depending on what you want
                    )
                last_name_frame_df=clean_dataframe(last_name_frame_df_raw)
                last_name_df = pd.concat([last_name_df,last_name_frame_df]) 
                
                get_counter+=1
                sleep_time=random_sleep(get_counter)       
                time.sleep(sleep_time)
            
            jewish_graves_last_name=((last_name_df['type']=='Jewish') & ((last_name_df['lastName']==last_name))).sum()
            non_jewish_graves_last_name=((last_name_df['type']=='Non_Jewish')  & ((last_name_df['lastName']==last_name))).sum()

            jewish_graves_last_name_frame=jewish_graves_last_name-last_jewish_graves_last_name
            non_jewish_graves_last_name_frame=non_jewish_graves_last_name-last_non_jewish_graves_last_name  
            
            last_jewish_graves_last_name=jewish_graves_last_name
            last_non_jewish_graves_last_name =non_jewish_graves_last_name           

            last_name_county_counts_df=pd.DataFrame([last_name,county_name,state_name,county_id,jewish_graves_last_name_frame,non_jewish_graves_last_name_frame]).T
            last_name_county_counts_df.columns=['last_name','county_name','state_name','county_id','jewish_last_name_count','non_jewish_last_name_count']
            
            last_name_counts_df=pd.concat([last_name_counts_df,last_name_county_counts_df])
            last_name_counts_df.to_csv(fileLastNamesOutInt,mode="w", index=False,header=True)
            
            print(f"    .... Last Name: {last_name:<10} • {county_name:<20} • {state_name:<15} • {county_id}  • Name Counts - Jewish/Non Jewish: {jewish_graves_last_name_frame:<4}/{non_jewish_graves_last_name_frame:<4}  • Cum: {jewish_graves_last_name:<4}/{non_jewish_graves_last_name:<4}")
        print(f"*** Last Name: {last_name:<10} • State: {state_name:<15} • Name Counts - Jewish/Non Jewish: {jewish_graves_last_name:<4}/{non_jewish_graves_last_name:<4}\n")
    last_name_counts_df.to_csv(fileLastNamesOut0,mode="w", index=False,header=True)
    last_name_counts_df.to_csv(fileLastNamesOut,mode="w", index=False,header=True)
    print("\n")
    print('Uploaded To: ',fileLastNamesOut0)
    print('***** Last Name Processing Complete *****' )      
        
    county_search_count=0
    first_name_df=pd.DataFrame([])
    first_name_counts_df=pd.DataFrame([])
    for first_name in first_name_list:
        print(f"•••• Searching: {first_name} • Search By: {search_type} • Search Before: {search_year}")
        last_jewish_graves_first_name=0
        last_non_jewish_graves_first_name =0
        county_count=0
        for county_id in county_id_list:
            county_name=county_list['county_name'].iloc[county_count]
            state_name=county_list['state_name'].iloc[county_count]
            county_count+=1
            frame_count=0
            grave_count=100000
            grave_frame_count=0
            frame_count=0
            while grave_frame_count<grave_count:
                frame_count+=1
                url=url0_f+str(grave_frame_count)+url1_f+first_name+url2_f+county_id+url3_f
                capture_json,grave_count=get_grave_count(url)
                first_name_frame_df_raw=extract_grave_data(capture_json)
                grave_frame_count+=len(first_name_frame_df_raw) 
                first_name_frame_df_raw['cemeteryId']=first_name_frame_df_raw['cemeteryId'].fillna(0).astype(int)
                first_name_frame_df_raw['lastName']=first_name_frame_df_raw['lastName'].astype(str).str.lower()
                first_name_frame_df_raw['firstName']=first_name_frame_df_raw['firstName'].astype(str).str.lower()
                first_name_frame_df_raw = pd.merge(
                        first_name_frame_df_raw,    
                        matched_cemeteries_df,
                        right_on='cemetery_id',             
                        left_on='cemeteryId',
                        how='inner'  # or 'left', 'right', 'outer' depending on what you want
                    )
                first_name_frame_df=clean_dataframe(first_name_frame_df_raw)
                first_name_df = pd.concat([first_name_df,first_name_frame_df]) 
                
                get_counter+=1
                sleep_time=random_sleep(get_counter)       
                time.sleep(sleep_time)
            
            jewish_graves_first_name=((first_name_df['type']=='Jewish') & ((first_name_df['firstName']==first_name))).sum()
            non_jewish_graves_first_name=((first_name_df['type']=='Non_Jewish')  & ((first_name_df['firstName']==first_name))).sum()
    
            jewish_graves_first_name_frame=jewish_graves_first_name-last_jewish_graves_first_name
            non_jewish_graves_first_name_frame=non_jewish_graves_first_name-last_non_jewish_graves_first_name  
            
            last_jewish_graves_first_name=jewish_graves_first_name
            last_non_jewish_graves_first_name =non_jewish_graves_first_name   
    
            
            first_name_county_counts_df=pd.DataFrame([first_name,county_name,state_name,county_id,jewish_graves_first_name_frame,non_jewish_graves_first_name_frame]).T
            first_name_county_counts_df.columns=['first_name','county_name','state_name','county_id','jewish_first_name_count','non_jewish_first_name_count']
            
            first_name_counts_df=pd.concat([first_name_counts_df,first_name_county_counts_df])
            first_name_counts_df.to_csv(fileFirstNamesOutInt,mode="w", index=False,header=True)
            
            print(f"    .... First Name: {first_name:<10} • {county_name:<20} • {state_name:<15} • {county_id}  • Name Counts - Jewish/Non Jewish: {jewish_graves_first_name_frame:<4}/{non_jewish_graves_first_name_frame:<4}  • Cum: {jewish_graves_first_name:<4}/{non_jewish_graves_first_name:<4}")
        print(f"*** First Name: {first_name:<10} • State: {state_name:<15} • Name Counts - Jewish/Non Jewish: {jewish_graves_first_name:<4}/{non_jewish_graves_first_name:<4}\n")
    first_name_counts_df.to_csv(fileFirstNamesOut0,mode="w", index=False,header=True)
    first_name_counts_df.to_csv(fileFirstNamesOut,mode="w", index=False,header=True)
    print("\n")
    print('Uploaded To: ',fileFirstNamesOut0)
    print('***** First Name Processing Complete *****' )  
   
 

**Starting ***  Extract Jewish ABMC Graves   *** , Time: 2025-06-01 08:16:49.778133  **

•••• Searching: baskind • Search By: birth • Search Before: 1920
    .... Last Name: baskind    • Camden County        • New Jersey      • county_1910  • Name Counts - Jewish/Non Jewish: 0   /0     • Cum: 0   /0   
    .... Last Name: baskind    • Delaware County      • Pennsylvania    • county_2263  • Name Counts - Jewish/Non Jewish: 1   /0     • Cum: 1   /0   
    .... Last Name: baskind    • Montgomery County    • Pennsylvania    • county_2286  • Name Counts - Jewish/Non Jewish: 0   /0     • Cum: 1   /0   
    .... Last Name: baskind    • Philadelphia County  • Pennsylvania    • county_2291  • Name Counts - Jewish/Non Jewish: 0   /0     • Cum: 1   /0   
*** Last Name: baskind    • State: Pennsylvania    • Name Counts - Jewish/Non Jewish: 1   /0   



Uploaded To:  C:\Users\jrozm\OneDrive\Benjamin\Analysis\SearchResults\Last_Name_Counts_Summary.csv
***** Last Name Processing Complete *****
•••• S

In [256]:
    county_search_count=0
    first_name_df=pd.DataFrame([])
    first_name_counts_df=pd.DataFrame([])
    for first_name in first_name_list:
        print(f"•••• Searching: {first_name} • Search By: {search_type} • Search Before: {search_year}")
        last_jewish_graves_first_name=0
        last_non_jewish_graves_first_name =0
        county_count=0
        for county_id in county_id_list:
            county_name=county_list['county_name'].iloc[county_count]
            state_name=county_list['state_name'].iloc[county_count]
            county_count+=1
            frame_count=0
            grave_count=100000
            grave_frame_count=0
            frame_count=0
            while grave_frame_count<grave_count:
                frame_count+=1
                url=url0_f+str(grave_frame_count)+url1_f+first_name+url2_f+county_id+url3_f
                capture_json,grave_count=get_grave_count(url)
                first_name_frame_df_raw=extract_grave_data(capture_json)
                grave_frame_count+=len(first_name_frame_df_raw) 
                first_name_frame_df_raw['cemeteryId']=first_name_frame_df_raw['cemeteryId'].fillna(0).astype(int)
                first_name_frame_df_raw['lastName']=first_name_frame_df_raw['lastName'].astype(str).str.lower()
                first_name_frame_df_raw['firstName']=first_name_frame_df_raw['firstName'].astype(str).str.lower()
                first_name_frame_df_raw = pd.merge(
                        first_name_frame_df_raw,    
                        matched_cemeteries_df,
                        right_on='cemetery_id',             
                        left_on='cemeteryId',
                        how='inner'  # or 'left', 'right', 'outer' depending on what you want
                    )
                first_name_frame_df=clean_dataframe(first_name_frame_df_raw)
                first_name_df = pd.concat([first_name_df,first_name_frame_df]) 
                
                get_counter+=1
                sleep_time=random_sleep(get_counter)       
                time.sleep(sleep_time)
            
            jewish_graves_first_name=((first_name_df['type']=='Jewish') & ((first_name_df['firstName']==first_name))).sum()
            non_jewish_graves_first_name=((first_name_df['type']=='Non_Jewish')  & ((first_name_df['firstName']==first_name))).sum()
    
            jewish_graves_first_name_frame=jewish_graves_first_name-last_jewish_graves_first_name
            non_jewish_graves_first_name_frame=non_jewish_graves_first_name-last_non_jewish_graves_first_name  
            
            last_jewish_graves_first_name=jewish_graves_first_name
            last_non_jewish_graves_first_name =non_jewish_graves_first_name   
    
            
            first_name_county_counts_df=pd.DataFrame([first_name,county_name,state_name,county_id,jewish_graves_first_name_frame,non_jewish_graves_first_name_frame]).T
            first_name_county_counts_df.columns=['first_name','county_name','state_name','county_id','jewish_first_name_count','non_jewish_first_name_count']
            
            first_name_counts_df=pd.concat([first_name_counts_df,first_name_county_counts_df])
            first_name_counts_df.to_csv(fileFirstNamesOutInt,mode="w", index=False,header=True)
            
            print(f"    .... First Name: {first_name:<10} • {county_name:<20} • {state_name:<15} • {county_id}  • Name Counts - Jewish/Non Jewish: {jewish_graves_first_name_frame:<4}/{non_jewish_graves_first_name_frame:<4}  • Cum: {jewish_graves_first_name:<4}/{non_jewish_graves_first_name:<4}")
        print(f"*** First Name: {first_name:<10} •  State: {state_name:<15} • Name Counts - Jewish/Non Jewish: {jewish_graves_first_name:<4}/{non_jewish_graves_first_name:<4}\n")
    first_name_counts_df.to_csv(fileFirstNamesOut0,mode="w", index=False,header=True)
    first_name_counts_df.to_csv(fileFirstNamesOut,mode="w", index=False,header=True)
    print("\n")
    print('Uploaded To: ',fileFirstNamesOut0)
    print('***** First Name Processing Complete *****' )  

•••• Searching: aaron • Search By: birth • Search Before: 1920
    .... First Name: aaron      • Bristol County       • Massachusetts   • county_1175  • Name Counts - Jewish/Non Jewish: 10  /75    • Cum: 10  /75  
    .... First Name: aaron      • Kent County          • Rhode Island    • county_2309  • Name Counts - Jewish/Non Jewish: 68  /19    • Cum: 78  /94  
    .... First Name: aaron      • Newport County       • Rhode Island    • county_2310  • Name Counts - Jewish/Non Jewish: 3   /19    • Cum: 81  /113 
    .... First Name: aaron      • Providence County    • Rhode Island    • county_2311  • Name Counts - Jewish/Non Jewish: 5   /87    • Cum: 86  /200 
*** First Name: aaron      •  State: Rhode Island    • Name Counts - Jewish/Non Jewish: 86  /200 

•••• Searching: abraham • Search By: birth • Search Before: 1920
    .... First Name: abraham    • Bristol County       • Massachusetts   • county_1175  • Name Counts - Jewish/Non Jewish: 63  /234   • Cum: 63  /234 
    .... First Nam

In [249]:
list(matched_cemeteries_df)

['type',
 'cemetery_id',
 'cemetery_name',
 'grave_count',
 'city',
 'state',
 'stateAbbreviation',
 'county',
 'state_id',
 'county_id',
 'cemetery_lat',
 'cemetery_lon',
 'County_Cemetery_Counts']

In [ ]:
url

In [ ]:

matched_cemeteries_df_header=['type','cemetery_id','cemetery_name','grave_count','city','state','stateAbbreviation',\
                              'county','state_id','county_id','cemetery_lat','cemetery_lon','County_Cemetery_Counts']
cluster_county_id=pd.DataFrame(county_id_list,columns=['county_id'])
# cluster_county_id.columns='county_id'
jewish_cemetery_df=pd.merge(
    jewish_cemetery_df,    
    cluster_county_id,
    right_on='county_id',             
    left_on='county_id',
    how='inner'  # or 'left', 'right', 'outer' depending on what you want
)
jewish_cemetery_df['type']='Jewish'

non_jewish_cemetery_df=pd.merge(
    non_jewish_cemetery_df,    
    cluster_county_id,
    right_on='county_id',             
    left_on='county_id',
    how='inner'  # or 'left', 'right', 'outer' depending on what you want
)
non_jewish_cemetery_df['type']='Non_Jewish'
matched_cemeteries_df=pd.DataFrame([])
matched_cemeteries_df = non_jewish_cemetery_df._append(jewish_cemetery_df, ignore_index = True) 
matched_cemeteries_df=matched_cemeteries_df[matched_cemeteries_df_header]
matched_cemeteries_df

In [ ]:
matched_cemeteries_df=pd.DataFrame([])
non_jewish_distance_df=pd.DataFrame([])

name_location=[name_lat,name_lon]
for non_jewish_index, non_jewish_row in non_jewish_cemetery_df.iterrows():
    distance=pd.Series(distance(name_location, (non_jewish_row['cemetery_lat'], non_jewish_row['cemetery_lon'])).miles)
    non_jewish_distance_element_df= pd.DataFrame([non_jewish_index['county_id'],pd.Series(distance(name_location, (non_jewish_row['cemetery_lat'], non_jewish_row['cemetery_lon'])).miles)])
    non_jewish_distance_df = pd.concat([non_jewish_distance_df, non_jewish_distance_elemen_df])
    print(non_jewish_distance_df)
    input('')
non_jewish_distance_df.columns=['distance']

non_jewish_cemetery_df['county_id'][non_jewish_distance_df['distance']<=jewish_radius]


In [ ]:
non_jewish_cemetery_df['county_id'][non_jewish_distance_df['distance']<=jewish_radius]

In [ ]:
non_jewish_cemetery_df=pd.read_csv(fileInN)